## Load Modules

In [28]:
import pandas as pd
import plotly.graph_objects as go
!pip install pvlib
from pvlib.pvsystem import PVSystem
from pvlib.location import Location
from pvlib.modelchain import ModelChain
from pvlib.temperature import TEMPERATURE_MODEL_PARAMETERS

## Set System Parameters

In [24]:
PV_Max_Power = 10000  # Maximum power in Watts
PV_Azimuth_deg = 180  # South-facing

## Load Data

In [25]:
# Load the dataset
url = 'https://raw.githubusercontent.com/OSU-fifem/OSU-ESE450/main/project/Yachats_Off_Grid_2023.csv'
df = pd.read_csv(url)

# Display column names so we can map them correctly if needed
print("Dataset Columns:", df.columns.tolist())

# Ensure everything is in UTC
df['Time'] = pd.to_datetime(df['Time'], utc=True)
df.set_index('Time', inplace=True)

display(df.head())

Dataset Columns: ['Time', 'Hour_of_Year', 'Hour_of_Day', 'Month', 'Temperature', 'Wind_Speed', 'Wind_Direction', 'DNI', 'DHI', 'GHI', 'Streamflow_m3_s', 'Base_Load_kW', 'Lighting_Plugs_kW', 'Induction_Range_kW', 'Total_Electrical_Load_kW', 'DHW_Demand_Liters']


,Hour_of_Year,Hour_of_Day,Month,Temperature,Wind_Speed,Wind_Direction,DNI,DHI,GHI,Streamflow_m3_s,Base_Load_kW,Lighting_Plugs_kW,Induction_Range_kW,Total_Electrical_Load_kW,DHW_Demand_Liters
Time,,,,,,,,,,,,,,,
2023-01-01 00:00:00+00:00,1,0,1,8.3,3.0,353,190,14,21,0.034,0.40,0.04,0.0,0.44,0.0
2023-01-01 01:00:00+00:00,2,1,1,8.1,3.0,352,0,0,0,0.031,0.40,0.06,0.0,0.46,0.0
2023-01-01 02:00:00+00:00,3,2,1,8.0,3.0,349,0,0,0,0.034,0.38,0.03,0.0,0.41,0.0
2023-01-01 03:00:00+00:00,4,3,1,7.9,2.9,349,0,0,0,0.032,0.39,0.06,0.0,0.45,0.0
2023-01-01 04:00:00+00:00,5,4,1,7.8,2.9,352,0,0,0,0.038,0.39,0.09,0.0,0.48,0.0


## Simulate PV

In [ ]:
# Prepare the weather dataframe for pvlib.
# You may need to tweak these column name strings to match the exact headers in the CSV printed above.
weather = pd.DataFrame()
weather['ghi'] = df.get('GHI', df.get('ghi', 0)) # Global Horizontal Irradiance
weather['dni'] = df.get('DNI', df.get('dni', 0)) # Direct Normal Irradiance
weather['dhi'] = df.get('DHI', df.get('dhi', 0)) # Diffuse Horizontal Irradiance
weather['temp_air'] = df.get('Temperature', df.get('temp_air', 20)) # Dry Bulb Temperature
weather['wind_speed'] = df.get('Wind Speed', df.get('wind_speed', 0))

# 1. Define Location
latitude = 41.31054
longitude = -124.09559
tz = 'UTC'
site = Location(latitude, longitude, tz=tz)

# 2. Define System (10 kW peak)
# We assume the array is tilted at the latitude angle and faces South (azimuth=180)
system = PVSystem(
    surface_tilt=latitude,
    surface_azimuth=PV_Azimuth_deg,
    # module_parameters are for the simple "PVWatts" model
    # gamma_pdc is the temperature coefficient of power, which we set to a typical value for silicon panels
    module_parameters={'pdc0': PV_Max_Power, 'gamma_pdc': -0.004},
    inverter_parameters={'pdc0': PV_Max_Power},
    # Use the SAPM temperature model parameters for an open rack glass-glass configuration, which is common for residential PV systems
    temperature_model_parameters=TEMPERATURE_MODEL_PARAMETERS['sapm']['open_rack_glass_glass']
)

# 3. Create ModelChain - Don't specify dc_model.  Infer it from module_parameters. 
mc = ModelChain(system, site, aoi_model='physical', spectral_model='no_loss')

# 4. Run Simulation
mc.run_model(weather)

# 5. Extract and Plot AC Power Output
ac_power = mc.results.ac

## Plot the Results

In [27]:
# Convert simulated AC power from Watts to kilowatts for comparison
ac_power_kw = ac_power / 1000

fig = go.Figure()

# Add Solar Production Trace
fig.add_trace(go.Scatter(
    x=ac_power_kw.index,
    y=ac_power_kw,
    mode='lines',
    name='Solar Production (kW)',
    line=dict(color='orange')
))

# Add Total Electrical Load Trace
fig.add_trace(go.Scatter(
    x=df.index,
    y=df['Total_Electrical_Load_kW'],
    mode='lines',
    name='Total Electrical Load (kW)',
    line=dict(color='blue')
))

# Update layout for better interactivity and labeling
fig.update_layout(
    title='Simulated 10kW Solar Array Production vs Total Electrical Load',
    xaxis_title='Time (UTC)',
    yaxis_title='Power (kW)',
    hovermode='x unified',
    template='plotly_white'
)

fig.show()